In [ ]:
!pip install langchain langchain_community neo4j langchain_groq pandas openpyxl  --quiet

In [ ]:
from google.colab import userdata

URI = userdata.get("NEO4J_URI")
USERNAME = userdata.get("NEO4J_USERNAME")
PASSWORD = userdata.get("NEO4J_PASSWORD")
groq_api_key= userdata.get("GROQ_API_KEY")

In [ ]:
from langchain_groq import ChatGroq
from neo4j import GraphDatabase

llm = ChatGroq(temperature=0.7, api_key=groq_api_key,model="llama3-8b-8192")

# Neo4j driver setup
driver = GraphDatabase.driver(URI, auth=(USERNAME, PASSWORD))

def retrieve_data():
    query = """
    MATCH (m:Member)-[r:PURCHASED]->(i:Item)
    RETURN m.id AS MemberID, i.name AS ItemPurchased, r.date AS PurchaseDate
    """

    with driver.session() as session:
        result = session.run(query)
        data = [{"MemberID": record["MemberID"], "ItemPurchased": record["ItemPurchased"], "PurchaseDate": record["PurchaseDate"]} for record in result]

    return data

In [ ]:
from langchain.chains import LLMChain
from langchain.prompts import PromptTemplate

def analyze_data(data, user_query):
    prompt_template = """
    You are an AI assistant that helps analyze purchase behavior data and identify patterns.
    Here's a list of purchases made by different members:
    {data}

    The user wants to know: {user_query}

    Based on this data, provide insights and analysis relevant to the user's query.
    """

    formatted_data = "\n".join([f"Member {record['MemberID']} bought {record['ItemPurchased']} on {record['PurchaseDate']}" for record in data])

    # Set up the prompt and LLM
    prompt = PromptTemplate(input_variables=["data", "user_query"], template=prompt_template)
    chain = LLMChain(llm=llm, prompt=prompt)

    # Generate the analysis
    analysis = chain.run(data=formatted_data, user_query=user_query)
    return analysis

In [ ]:
data = retrieve_data()

user_query = input("How can I help you?")
analysis = analyze_data(data, user_query)

print("\nAnalysis based on your query:")
print(analysis)

# Close the driver
driver.close()